# Selfwork RAG Base

Esempio base di RAG (Retrieval Augmented Generation): i curriculum nella cartella `resumes` vengono divisi in paragrafi, trasformati in embeddings e salvati in ChromaDB. Data una richiesta dell'utente, si recupera il paragrafo più simile e lo si passa come contesto al modello di linguaggio, che indica il candidato più adatto.

### ChromaDB

In [ ]:
%%capture --no-stderr
%pip install chromadb openai python-dotenv

ChromaDB è un database progettato per salvare e cercare embeddings, cioè rappresentazioni numeriche dei testi. Permette di trovare i documenti semanticamente più vicini a una domanda.

https://docs.trychroma.com/docs/overview/introduction

### Chiave OpenAI

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('.env')

openai_key = os.environ['OPENAI_API_KEY']

### Lettura dei documenti

In [ ]:
documents_dir = "resumes"
documents = []
metadatas = []
ids = []

id = 0

for filename in os.listdir(documents_dir):
    if filename.endswith(".txt"):
        with open(os.path.join(documents_dir, filename), 'r', encoding='utf-8') as file:
            chunks = file.read().replace('\n', '.').split('### ')
            for chunk in chunks:
                if not chunk.isspace() and not chunk == "":
                    documents.append(chunk)
                    metadatas.append({"source": filename, "info": chunks[1]})
                    ids.append(str(id))
                    id += 1

print(f"Paragrafi letti: {len(documents)}")

In [ ]:
metadatas[0:15]

In [ ]:
metadatas[0]

### OpenAI Embeddings

In [ ]:
from chromadb.utils import embedding_functions

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small"
)

In [ ]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="CVs",
    embedding_function=openai_ef
)

In [ ]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

In [ ]:
collection.get()

### Vector Similarity

In [ ]:
user_question = "mi serve qualcuno per promuovere il mio prodotto"

results = collection.query(
    query_texts=[user_question],
    n_results=1
)

results

In [ ]:
results['metadatas'][0][0]['info']

### Prepariamo il prompt con il contesto

In [ ]:
context = f"CONTESTO: nome file {results['metadatas'][0][0]['source']} ecco il paragrafo piu' significativo: {results['documents'][0][0]} ricorda sempre di menzionare il nome del candidato all'inizio e i dati personali alla fine per il contatto, ti lascio tutto qui: {results['metadatas'][0][0]['info']}"

context

In [ ]:
prompt = f"""Dato il seguente contesto {context} rispondi alla domanda dell'utente {user_question}
spiegando che nel file individuato c'e' il profilo piu' adatto.
Argomenta la scelta utilizzando il contenuto del testo individuato nel contesto
"""

prompt

### Utilizziamo un modello di linguaggio

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=openai_key)

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "developer",
            "content": "Sei un assistente HR, specializzato nella ricerca di profili professionali"
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(completion.choices[0].message.content)